In [19]:
# conda env 'at3d_pure'

import sys
sys.path.append('/data/keeling/a/johntl3/b/LundstromPub2025/src')

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import at3d
import netCDF4 as nc
import os
from collections import OrderedDict
from util import load_sensor_dict
import yaml

In [22]:
config = {}

## INPUT
config['obs_file'] = '/data/keeling/a/johntl3/b/LundstromPub2025/CAMP2Ex_RF17/intermediate_data/AT3D_OBS_CAMP2Ex_RF17_default_1400.nc'
config['rte_grid_file'] = '/data/keeling/a/johntl3/b/LundstromPub2025/CAMP2Ex_RF17/intermediate_data/RTG_1400.nc'
config['background_scatterers_file'] = '/data/keeling/a/johntl3/b/LundstromPub2025/CAMP2Ex_RF17/intermediate_data/MIXTURE_default.nc'
config['cloud_volume_mask_file'] = '/data/keeling/a/johntl3/b/LundstromPub2025/CAMP2Ex_RF17/intermediate_data/CVM_all_cloud_1400.nc'
config['config_file_name'] = '/data/keeling/a/johntl3/b/LundstromPub2025/configs/misr_config.json'
config['surface_wind_speed'] = 1.33 # m/s
# Reff(z) = a*z + b
config['a'] = 4.292 # um / km
config['b'] = 2.623 # um
config['error_cov_sf'] = 1e7

## Restart From
config['restart_iter'] = '/data/keeling/a/johntl3/b/LundstromPub2025/CAMP2Ex_RF17/AT3D_output/V02/CVM/unconstrained_darksurf/iterations/iter033.nc'

## OUTPUT
config['out_path'] = '/data/keeling/a/johntl3/b/LundstromPub2025/CAMP2Ex_RF17/AT3D_output/V02/CVM/unconstrained_darksurf_restart/'
iter_dir = '{}iterations/'.format(config['out_path'])
supp_file = '{}loss.csv'.format(config['out_path'])

# Define output files
os.makedirs(iter_dir, exist_ok=True)
assert len(os.listdir(iter_dir)) == 0, "Iterations already saved in {}.\nSpecify a new output location.".format(iter_dir)

with open('{}config.yml'.format(config['out_path']), 'w') as file:
    yaml.dump(config, file, sort_keys=False)


In [7]:

# Mie table directory
mie_dir = '/data/keeling/a/johntl3/b/LundstromPub2025/mie_tables/'

rte_grid = xr.load_dataset(rte_grid_file)
obs = load_sensor_dict(obs_file)

# add sensor uncertainty model
inverse_covariance = error_cov_sf * np.diag(np.ones(4))
uncertainty = at3d.uncertainties.Uncertainty(inverse_covariance, 'L2')
obs.add_uncertainty_model('MISR', uncertainty)

forward_sensors = obs.make_forward_sensors()
wavelength = forward_sensors.get_unique_solvers()[0]

# Surface wind speed comes from ERA5 reanalysis - 
# see ./ancillary/CAMP2Ex_RF17_ERA5_Surface.ipynb
surfaces = OrderedDict()
m_water = 1.33197 - 1j*2.41e-8
surfaces[wavelength] = at3d.surface.wave_fresnel(np.real(m_water), np.imag(m_water), surface_wind_speed)

numerical_parameters = OrderedDict()
numerical_parameters[wavelength] = at3d.configuration.get_config(config_file_name)

sources = OrderedDict()
sources[wavelength] = at3d.source.solar(
    wavelength,
    -1*obs['MISR']['sensor_list'][0].mu0,
    obs['MISR']['sensor_list'][0].SAA,
    solarflux=1.0,
    skyrad=0.0
)

background_optical_scatterers = OrderedDict()
background_properties = xr.load_dataset(background_scatterers_file)
background_optical_scatterers[wavelength] = {}
background_optical_scatterers[wavelength]['background'] = background_properties

mie_mono_table = at3d.mie.get_mono_table(
    'Water',
    (wavelength, wavelength),
    max_integration_radius=65.0,
    minimum_effective_radius=5.5,
    relative_dir=mie_dir,
    verbose=False
)
mie_mono_tables = OrderedDict()
mie_mono_tables[wavelength] = mie_mono_table
# mie_mono_table.to_netcdf('{}red_mie_table.nc'.format(mie_dir))

cloud_optical_prop_gen = at3d.medium.OpticalPropertyGenerator(
    'cloud',
    mie_mono_tables,
    at3d.size_distribution.gamma,
    particle_density=1.0,
    maxnphase=None,
    interpolation_mode='exact',
    density_normalization='density',
    reff=np.linspace(1, 30, 20),
    veff=np.linspace(0.01, 0.2, 5)
)

cloud = rte_grid.copy(deep=True)
cloud['density'] = (['x','y','z'], np.zeros((rte_grid.dims['x'], rte_grid.dims['y'], rte_grid.dims['z'])))
reff_profile = a * rte_grid.z.data + b
reff = np.tile(reff_profile, (rte_grid.dims['x'], rte_grid.dims['y'], 1))
cloud['reff'] = (['x','y','z'], reff)
cloud['veff'] = (['x','y','z'], np.ones(cloud.density.shape) * 0.1)
cloud_optical_properties = cloud_optical_prop_gen(cloud)

grid_to_optical_properties = at3d.medium.GridToOpticalProperties(
    rte_grid,
    'cloud',
    wavelength,
    fixed_dataset=cloud_optical_properties[wavelength].drop_vars('extinction')
)

cloud_volume_mask = xr.load_dataset(cloud_volume_mask_file)['cloud_volume_mask'].data
# cloud_volume_mask[0,:,:] = cloud_volume_mask[-1,:,:] = cloud_volume_mask[:,0,:] = cloud_volume_mask[:,-1,:] = 0.0
extinction_coordinate_transform = None
extinction_state2grid_transform = at3d.transforms.StateToGridMask(mask=cloud_volume_mask)
extinction_transforms = (extinction_coordinate_transform, extinction_state2grid_transform)

cloud_unknown_scatterer = at3d.medium.UnknownScatterer(grid_to_optical_properties, extinction=extinction_transforms)
unknown_scatterers = at3d.containers.UnknownScatterers(cloud_unknown_scatterer)

num_stokes = OrderedDict()
num_stokes[wavelength] = 1

solvers = at3d.containers.SolversDict()
state_gen = at3d.medium.StateGenerator(
    solvers,
    unknown_scatterers,
    surfaces,
    numerical_parameters,
    sources,
    background_optical_scatterers,
    num_stokes
)
min_bounds, max_bounds = state_gen.transform_bounds()


In [8]:
if(restart_iter!=None):
    print('restart')
    restart_sensor, restart_solver, restart_rte_grid = at3d.util.load_forward_model(restart_iter)
    extinction_init = restart_solver[wavelength].medium['cloud'].extinction.data
else:
    print('no restart')
    extinction_init = cloud_volume_mask.astype(float)*0.1
extinction_init_1d = state_gen._unknown_scatterers['cloud'].variables['extinction'].state_to_grid.inverse_transform(extinction_init)
x0 = state_gen._unknown_scatterers['cloud'].variables['extinction'].coordinate_transform.inverse_transform(extinction_init_1d)
state_gen(x0)


no restart


In [7]:
forward_sensors.get_measurements(solvers)


  ! Iter Log(Sol)  SplitCrit  Npoints  Nsh(avg)   [Radiance 0.670 micron]
     1  -0.854  0.420E+00    67600   256.00  1.000   [Radiance 0.670 micron]
     2  -1.552  0.422E+00    67600   256.00  1.000   [Radiance 0.670 micron]
     3  -2.010  0.422E+00    67600   256.00  1.000   [Radiance 0.670 micron]
     4  -2.119  0.177E+00    69809   256.00  1.000   [Radiance 0.670 micron]
     5  -2.006  0.103E+00    78732   256.00  1.000   [Radiance 0.670 micron]
     6  -2.188  0.872E-01    84645   256.00  1.000   [Radiance 0.670 micron]
     7  -2.271  0.684E-01    96824   256.00  1.000   [Radiance 0.670 micron]
     8  -2.325  0.564E-01   112150   256.00  1.000   [Radiance 0.670 micron]
     9  -2.406  0.488E-01   126122   256.00  1.000   [Radiance 0.670 micron]
    10  -2.457  0.424E-01   142655   256.00  1.000   [Radiance 0.670 micron]
    11  -2.415  0.377E-01   159244   256.00  1.000   [Radiance 0.670 micron]
    12  -2.620  0.369E-01   162817   256.00  1.000   [Radiance 0.670 micron]
  

In [ ]:
parallel_solve_kwargs = {
    'n_jobs': 1,
    'mpi_comm': None, 
    'verbose': True, 
    'maxiter': 100, 
    'init_solution': True
}

objective_function = at3d.optimize.ObjectiveFunction.LevisApproxUncorrelatedL2(
    obs,
    solvers,
    forward_sensors,
    unknown_scatterers,
    state_gen,
    state_gen.project_gradient_to_state,
    parallel_solve_kwargs=parallel_solve_kwargs,
    gradient_kwargs={'cost_function': 'L2', 'exact_single_scatter':True},
    uncertainty_kwargs={'add_noise':False},
    min_bounds=min_bounds, 
    max_bounds=max_bounds
)

def save(optimizer):
    if(optimizer.iteration == None):
        outFile = '{}iter{:03d}.nc'.format(iter_dir, 0)
    else:
        outFile = '{}iter{:03d}.nc'.format(iter_dir, optimizer.iteration)
    at3d.util.save_forward_model(outFile, forward_sensors, solvers)
    with open(supp_file, mode='a') as file:
        file.write('{}, {}, {}\n'.format(optimizer.iteration, optimizer.objective_fn.loss, optimizer.objective_fn._total_obj_fn_time))
    return {'None': None}
callback = at3d.callback.CallbackFn(save)


In [ ]:

with open(supp_file, mode='a') as file:
    file.write('iteration, loss, total_obj_fn_time\n')

options = {'maxiter': 100, 'maxls': 25, 'disp': True, 'gtol': 1e-16, 'ftol': 1e-8}
optimizer = at3d.optimize.Optimizer(objective_function, callback_fn=callback, options=options)

# Save initial conditions
save(optimizer)

result = optimizer.minimize(x0)

outFile = '{}iter{:03d}.nc'.format(iter_dir, optimizer.iteration+1)
at3d.util.save_forward_model(outFile, forward_sensors, solvers)



